# Item

> File type icons, cell render callback for virtual collection, and empty/error state components.

In [ ]:
#| default_exp components.item

In [ ]:
#| export
from typing import Any, Callable, Dict, Optional

from fasthtml.common import Div, Span, P, Input

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_input.checkbox import checkbox, checkbox_colors, checkbox_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, min_h, min_w
from cjm_fasthtml_tailwind.utilities.typography import font_size, truncate
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Lucide icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Local imports
from cjm_file_discovery.core.models import FileInfo, FileType
from cjm_fasthtml_file_browser.core.config import FileBrowserConfig, FileColumn

# Virtual collection
from cjm_fasthtml_virtual_collection.core.models import CellRenderContext

## Icon Mapping

Maps file types to Lucide icon names for consistent iconography.

In [ ]:
#| export
FILE_TYPE_ICONS: Dict[FileType, str] = {
    FileType.AUDIO: "file-music",
    FileType.VIDEO: "file-video-camera",
    FileType.IMAGE: "image",
    FileType.DOCUMENT: "file-text",
    FileType.CODE: "file-code",
    FileType.DATA: "database",
    FileType.ARCHIVE: "archive",
    FileType.OTHER: "file",
}

BROWSER_ICONS: Dict[str, str] = {
    "folder": "folder",
    "folder_open": "folder-open",
    "parent": "arrow-up",
    "home": "house",
    "refresh": "refresh-cw",
    "list_view": "list",
    "grid_view": "grid-3x3",
    "sort_asc": "arrow-up-narrow-wide",
    "sort_desc": "arrow-down-wide-narrow",
    "check": "check",
    "x": "x",
}

In [ ]:
# Test icon mapping
assert FILE_TYPE_ICONS[FileType.CODE] == "file-code"
assert FILE_TYPE_ICONS[FileType.AUDIO] == "file-music"
assert BROWSER_ICONS["folder"] == "folder"
print("Icon mapping tests passed!")

Icon mapping tests passed!


## Icon Rendering

In [ ]:
#| export
def _get_file_icon(
    file_info: FileInfo,  # File to get icon for
    size: int = 4         # Icon size (Tailwind scale)
) -> Any:  # Lucide icon component
    """Get the appropriate icon for a file based on its type."""
    if file_info.is_directory:
        icon_name = BROWSER_ICONS["folder"]
        color_cls = str(text_dui.warning)
    else:
        icon_name = FILE_TYPE_ICONS.get(file_info.file_type, "file")
        # Color based on file type
        color_map = {
            FileType.CODE: str(text_dui.info),
            FileType.DATA: str(text_dui.primary),
            FileType.IMAGE: str(text_dui.success),
            FileType.AUDIO: str(text_dui.secondary),
            FileType.VIDEO: str(text_dui.accent),
            FileType.DOCUMENT: str(text_dui.base_content),
            FileType.ARCHIVE: str(text_dui.warning),
        }
        color_cls = color_map.get(file_info.file_type, str(text_dui.base_content))
    
    return lucide_icon(icon_name, size=size, cls=color_cls)

In [ ]:
from fasthtml.common import to_xml

# Test icon rendering - verify icons are generated without errors
folder = FileInfo(name="test", path="/test", is_directory=True)
icon = _get_file_icon(folder)
html = to_xml(icon)
assert "<svg" in html.lower()  # SVG element present
assert "text-warning" in html  # Folder color class

py_file = FileInfo(name="test.py", path="/test.py", is_directory=False, file_type=FileType.CODE)
icon = _get_file_icon(py_file)
html = to_xml(icon)
assert "<svg" in html.lower()
assert "text-info" in html  # Code file color class

print("Icon rendering tests passed!")

Icon rendering tests passed!


## Cell Render Callback

The `create_file_cell_renderer` factory returns a `render_cell` callback for the virtual collection.
The callback dispatches on `ctx.column.key` to render each column's content.

In [ ]:
#| export
def create_file_cell_renderer(
    config: FileBrowserConfig,  # Browser config (for selection state access)
    get_selection: Callable,    # () -> BrowserSelection, returns current selection
    select_url: str = "",       # URL for checkbox click selection route
) -> Callable:  # render_cell(item: FileInfo, ctx: CellRenderContext) -> Any
    """Create a render_cell callback for the virtual collection.

    Returns a closure that renders cell content based on column key.
    """
    from cjm_fasthtml_file_browser.core.models import BrowserSelection

    def render_cell(
        item: FileInfo,          # File item for this row
        ctx: CellRenderContext,  # Cell context from virtual collection
    ) -> Any:  # Cell content (not the container div — VC wraps it)
        """Render cell content for a file browser row."""
        key = ctx.column.key

        if key == "select":
            selection = get_selection()
            is_selected = selection.is_selected(item.path)
            can_select = config.can_select(item)
            if not can_select:
                return Span()
            attrs = {
                "type": "checkbox",
                "checked": is_selected,
                "autocomplete": "off",
                "cls": combine_classes(checkbox, checkbox_colors.primary, checkbox_sizes.sm),
                "onclick": "event.stopPropagation()",
            }
            if select_url:
                attrs["hx_get"] = f"{select_url}?path={item.path}"
                attrs["hx_swap"] = "none"
            return Input(**attrs)

        elif key == "name":
            return Div(
                Span(_get_file_icon(item, size=4), cls=str(m.r(2))),
                Span(item.name, cls=combine_classes(truncate, grow())),
                cls=combine_classes(flex_display, items.center, min_w._0)
            )

        elif key == "size":
            text = item.size_str if not item.is_directory else "\u2014"
            return Span(text, cls=combine_classes(text_dui.base_content.opacity(70), font_size.sm))

        elif key == "modified":
            return Span(
                item.modified_str,
                cls=combine_classes(text_dui.base_content.opacity(70), font_size.sm),
            )

        elif key == "type":
            text = "Folder" if item.is_directory else (item.extension or "File").upper()
            return Span(text, cls=combine_classes(text_dui.base_content.opacity(70), font_size.sm))

        return Span("")

    return render_cell

In [ ]:
from cjm_fasthtml_virtual_collection.core.models import ColumnDef
from cjm_fasthtml_file_browser.core.models import BrowserSelection

config = FileBrowserConfig()
selection = BrowserSelection()
render_cell = create_file_cell_renderer(config, lambda: selection, select_url="/sel")

# Test name column
name_col = ColumnDef(key="name", header="Name")
folder = FileInfo(name="Documents", path="/docs", is_directory=True)
ctx = CellRenderContext(row_index=0, column=name_col, total_items=5)
html = to_xml(render_cell(folder, ctx))
assert "Documents" in html
assert "<svg" in html.lower()

# Test size column
size_col = ColumnDef(key="size", header="Size")
f = FileInfo(name="a.txt", path="/a.txt", is_directory=False, size=1024)
ctx = CellRenderContext(row_index=0, column=size_col, total_items=5)
html = to_xml(render_cell(f, ctx))
assert "1.0 KB" in html or "1 KB" in html

# Test select column — file is selectable
sel_col = ColumnDef(key="select", header="")
ctx = CellRenderContext(row_index=0, column=sel_col, total_items=5)
html = to_xml(render_cell(f, ctx))
assert "checkbox" in html
assert "/sel" in html

# Test select column — folder not selectable (default config: files only)
html = to_xml(render_cell(folder, ctx))
assert "checkbox" not in html

# Test modified column
mod_col = ColumnDef(key="modified", header="Modified")
ctx = CellRenderContext(row_index=0, column=mod_col, total_items=5)
html = to_xml(render_cell(f, ctx))
assert isinstance(html, str)

print("Cell render callback tests passed!")

## Empty and Error States

Components displayed by the virtual collection when a directory is empty or inaccessible.

In [ ]:
#| export
def render_empty_state(
    message: str = "No files found",  # Message to display
    icon_name: str = "folder-open"    # Lucide icon name
) -> Any:  # Empty state component
    """Render empty state for when a directory has no matching items."""
    return Div(
        Div(
            lucide_icon(icon_name, size=12, cls=str(text_dui.base_content.opacity(30))),
            cls=str(m.b(4))
        ),
        P(message, cls=combine_classes(text_dui.base_content.opacity(50), font_size.lg)),
        cls=combine_classes(
            flex_display, flex_direction.col, items.center, justify.center,
            p(8), min_h(48)
        )
    )


def render_error_state(
    error_message: str  # Error message to display
) -> Any:  # Error state component
    """Render error state for when directory access fails."""
    return Div(
        Div(
            lucide_icon("circle-alert", size=12, cls=str(text_dui.error)),
            cls=str(m.b(4))
        ),
        P(error_message, cls=combine_classes(text_dui.error, font_size.lg)),
        cls=combine_classes(
            flex_display, flex_direction.col, items.center, justify.center,
            p(8), min_h(48)
        )
    )

In [ ]:
# Test empty state
empty = render_empty_state("This folder is empty")
html = to_xml(empty)
assert "This folder is empty" in html
assert "<svg" in html.lower()

# Test error state
error = render_error_state("Permission denied")
html = to_xml(error)
assert "Permission denied" in html
assert "text-error" in html

print("Empty/error state tests passed!")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()